# 欢迎来到第二周！

## 前沿模型 API

## 练习目标（理念）

第一周我们主要在 **聊天 UI** 里试用多个 Frontier LLM，并接通过 OpenAI API。  
今天改为：**直接用各家 API / OpenAI 兼容端点** 调用模型，并在后半部分做一个**三方聊天机器人**对话实验。

## 和本课概念的关系

| 本课概念 | 本笔记本里你会看到 |
|----------|-------------------|
| 多厂商 API | OpenAI / Anthropic / Gemini / Groq / DeepSeek / Grok / OpenRouter |
| OpenAI 兼容 `base_url` | 一套 SDK，换地址与密钥 |
| 推理强度 `reasoning_effort` | 同一谜题不同思考预算 |
| 提示缓存（Prompt Caching） | LiteLLM + 长上下文 Hamlet |
| 多角色对话历史 | 两个 / 三个 chatbot 互相对话 |

## 怎么跑

1. 准备 `.env`（可选多家密钥；至少 OpenAI 或本地 Ollama）
2. 从上到下运行；没有的密钥对应单元格会失败，可跳过
3. 后半「三方对话」需要本地 Ollama 已拉取相应模型


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">重要说明 —— 请先读</h2>
            <span style="color:#900;">
                课程实验会持续更新、补充示例与练习。
                每周开始时建议先 <code>git pull</code>，再按需合并你的本地改动。
                有合并疑问可问 ChatGPT，或联系课程助教/讲师。
            </span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">资源页提醒</h2>
            <span style="color:#f71;">
                课程资源（含幻灯片）入口：<br/>
                <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
                建议收藏；后续还会继续补充链接。
            </span>
        </td>
    </tr>
</table>


## 设置 API 密钥（可选）

接下来会向多家模型提问。密钥**完全可选**：有 Anthropic / Gemini 等就可以加上；不想额外花钱，也可以只看演示或改走本地 Ollama。

常用控制台：

- OpenAI：https://openai.com/api/
- Anthropic：https://console.anthropic.com/
- Google：https://aistudio.google.com/
- DeepSeek：https://platform.deepseek.com/
- Groq：https://console.groq.com/
- Grok：https://console.x.ai/
- OpenRouter（多家统一入口）：https://openrouter.ai/

一般步骤：到计费页充值（部分有免费额度）→ 到 API Keys 页复制密钥。

### 写入 `.env`

```
OPENAI_API_KEY=xxxx
ANTHROPIC_API_KEY=xxxx
GOOGLE_API_KEY=xxxx
DEEPSEEK_API_KEY=xxxx
GROQ_API_KEY=xxxx
GROK_API_KEY=xxxx
OPENROUTER_API_KEY=xxxx
```

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">每次改完 .env</h2>
            <span style="color:#900;">记得保存，并重新运行 <code>load_dotenv(override=True)</code>。</span>
        </td>
    </tr>
</table>


In [ ]:
# ========== 导入：多模型实验要用的工具箱 ==========

# os：读环境变量（Environment Variables）里的各家 API Key
import os
# requests：后面探测本地 Ollama 是否在跑
import requests
# load_dotenv：把 .env 密钥读进进程，避免写进笔记本
from dotenv import load_dotenv
# OpenAI 客户端：官方 OpenAI，也用于各家 OpenAI 兼容端点
from openai import OpenAI
# Markdown + display：在笔记本里漂亮展示模型回复
from IPython.display import Markdown, display


In [ ]:
# ========== 加载密钥：从环境变量取出各家 API Key 并做存在性检查 ==========

# override=True：以 .env 覆盖进程里已有同名变量
load_dotenv(override=True)
# 逐个读取；没有则为 None
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

# 打印前几位：确认「有密钥」且没贴错，又不暴露全部（文案保持英文）
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
# ========== 客户端：默认 OpenAI + 各家兼容端点 ==========

# 连接到 OpenAI 客户端库
# 围绕 HTTP 端点调用的薄包装器

# 默认读环境变量 OPENAI_API_KEY
openai = OpenAI()

# 对于 Gemini、DeepSeek 和 Groq，我们可以使用 OpenAI python 客户端
# 因为 Google 和 DeepSeek 拥有与 OpenAI 兼容的端点
# OpenAI 允许您更改 base_url

# 各家 OpenAI 兼容（或近似兼容）的 base_url —— URL 字符串勿改
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

# 同一个 OpenAI() 类，换 api_key + base_url 就能打到不同后端
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
# 本地 Ollama：api_key 多为占位
ollama = OpenAI(api_key="ollama", base_url=ollama_url)


In [ ]:
# ========== 共用 messages：让各家模型讲同一个笑话 ==========

# messages 列表：这里只有一条 user；prompt 保留英文
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]


In [ ]:
# ========== 调用 OpenAI：gpt-4.1-mini 讲笑话 ==========

# chat.completions.create：标准 Chat Completions
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=tell_a_joke)
# 取出助手文本，用 Markdown 展示
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 调用 Anthropic（经 OpenAI 兼容客户端） ==========

# 模型 id 必须与 Anthropic 当前可用名一致；勿改字符串
response = anthropic.chat.completions.create(model="claude-sonnet-4-5-20250929", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))


## 训练时缩放 vs 推理时缩放（Inference-time Scaling）

同一道题，可以换：

- **更大 / 更强的模型**（训练侧能力）
- 或同一模型提高 **`reasoning_effort`**（推理时多想一会儿）

下面用一枚「两枚硬币」概率谜题做对比。


In [ ]:
# ========== 简单谜题 messages：只要概率数字答案 ==========

# prompt 保留英文，保证实验可复现
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]


In [ ]:
# ========== gpt-5-nano + reasoning_effort=minimal：少思考 ==========

response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== gpt-5-nano + reasoning_effort=low：稍多思考 ==========

response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 换更强小模型 gpt-5-mini，仍用 minimal ==========

response = openai.chat.completions.create(model="gpt-5-mini", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))


## 测试「更强」的模型

下面换成更难的「书架与书虫」谜题，对比不同厂商/档位模型的推理表现。


In [ ]:
# ========== 难题文本 → hard_puzzle messages ==========
# prompt 保留英文，保证实验可复现

# 三引号字符串：难题题干（发给模型，保留英文）
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
# 包装成标准 Chat Completions 的 user 消息列表
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [ ]:
# ========== 难题：gpt-5-nano + minimal ==========

response = openai.chat.completions.create(model="gpt-5-nano", messages=hard_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 难题：Claude Sonnet ==========

response = anthropic.chat.completions.create(model="claude-sonnet-4-5-20250929", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 难题：gpt-5（更强档） ==========

response = openai.chat.completions.create(model="gpt-5", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 难题：Gemini 2.5 Pro（兼容端点） ==========

response = gemini.chat.completions.create(model="gemini-2.5-pro", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))


## 博弈挑战：Steal or Share？

用「囚徒困境」式游戏节目设定，看不同模型更倾向合作还是背叛。  
（提示词保留英文，便于公平对比。）


In [ ]:
# ========== 困境 prompt → dilemma messages ==========
# Steal/Share 设定保留英文，便于跨模型公平对比

# 囚徒困境式游戏节目设定（prompt 保留英文）
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

# 打成 messages，供多家模型同一输入对比
dilemma = [
    {"role": "user", "content": dilemma_prompt},
]


In [ ]:
# ========== 困境：Claude ==========

response = anthropic.chat.completions.create(model="claude-sonnet-4-5-20250929", messages=dilemma)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 困境：Groq 上的 openai/gpt-oss-120b ==========

response = groq.chat.completions.create(model="openai/gpt-oss-120b", messages=dilemma)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 困境：DeepSeek Reasoner ==========

response = deepseek.chat.completions.create(model="deepseek-reasoner", messages=dilemma)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 困境：Grok-4 ==========

response = grok.chat.completions.create(model="grok-4", messages=dilemma)
display(Markdown(response.choices[0].message.content))


## 本地化（Ollama）

把 `OpenAI` 客户端的 `base_url` 指到 `http://localhost:11434/v1`，就能用同一套 SDK 调本地模型。


In [ ]:
# ========== 探测 Ollama 是否在本地监听 ==========

# 根路径应有响应；若连接失败，请先在终端启动 ollama serve
requests.get("http://localhost:11434/").content

# 如果未运行，请在命令行运行 ollama serve


In [ ]:
# 拉取较小本地模型 llama3.2（首次会下载权重）
!ollama pull llama3.2


In [ ]:
# 仅当机器内存较大时再跑（建议至少 16GB RAM）
# 拉取更大的 gpt-oss:20b
!ollama pull gpt-oss:20b


In [ ]:
# ========== 本地 llama3.2 解 easy_puzzle ==========

response = ollama.chat.completions.create(model="llama3.2", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 本地 gpt-oss:20b 解 easy_puzzle ==========

response = ollama.chat.completions.create(model="gpt-oss:20b", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))


## Gemini 与 Anthropic 官方客户端库

前面多用 OpenAI Python 客户端打兼容端点；各厂商也有**自家 SDK**。下面各演示一次原生调用写法。


In [ ]:
# ========== Google 官方 genai 客户端 ==========

# 从 google.genai 导入（依赖已安装且 GOOGLE_API_KEY 可用）
from google import genai

# 默认从环境变量读取凭证
client = genai.Client()

# generate_content：Gemini 原生接口（不是 chat.completions）
response = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
# 原生响应用 .text 取文本
print(response.text)


In [ ]:
# ========== Anthropic 官方 SDK ==========

from anthropic import Anthropic

# 默认读 ANTHROPIC_API_KEY
client = Anthropic()

# messages.create：Anthropic 原生消息 API（字段与 OpenAI 略有不同）
response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    messages=[{"role": "user", "content": "Describe the color Blue to someone who's never been able to see in 1 sentence"}],
    # 限制最长生成，控制费用与篇幅
    max_tokens=100
)
# 内容在 content 块列表里；取第一块的 text
print(response.content[0].text)


## 路由层与抽象层

从 [OpenRouter.ai](https://openrouter.ai/) 开始：一个入口可路由到上面提到的许多模型。

打开网站浏览模型列表。下面试用中国创业公司 z.ai 的 **GLM 4.5**（此前还没在本课直接出现）。


In [ ]:
# ========== 经 OpenRouter 调用 z-ai/glm-4.5 ==========

response = openrouter.chat.completions.create(model="z-ai/glm-4.5", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))


## LangChain（功能强，也偏「重」）

先快速看一眼 LangChain 的 `ChatOpenAI`：用 `invoke` 发同一条 `tell_a_joke`。


In [ ]:
# ========== LangChain ChatOpenAI 封装 ==========

from langchain_openai import ChatOpenAI

# 指定模型；内部仍走 OpenAI 生态
llm = ChatOpenAI(model="gpt-5-mini")
# invoke：同步调用，传入 messages 列表
response = llm.invoke(tell_a_joke)

# LangChain 消息对象用 .content 取文本
display(Markdown(response.content))


## LiteLLM（轻量统一调用）

作者偏好的轻量抽象：一个 `completion(...)` 接口，用 `provider/model` 字符串路由。


In [ ]:
# ========== LiteLLM completion：统一入口 ==========

from litellm import completion
# model 写成 openai/gpt-4.1 这种「厂商/模型」形式
response = completion(model="openai/gpt-4.1", messages=tell_a_joke)
# 响应形状接近 OpenAI
reply = response.choices[0].message.content
display(Markdown(reply))


In [ ]:
# ========== 打印 token 用量与费用（LiteLLM hidden params） ==========

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

## 专业能力：提示缓存（Prompt Caching）

用 LiteLLM + 长文本（整本 Hamlet）观察：把大段静态上下文放进提示后，**再次请求**时缓存命中如何影响费用与 cached tokens。


In [ ]:
# ========== 读入 hamlet.txt，并定位一段台词 ==========

# 以 UTF-8 打开同目录（或 cwd）下的 hamlet.txt
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

# 找到台词起点，打印后 100 字符做抽查
loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])


In [ ]:
# ========== 短问题（尚无全文上下文） ==========

question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]


In [ ]:
# ========== 无全文时问 Gemini：可能靠参数记忆/猜测 ==========

response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 基线用量：还没有超长前缀 ==========

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

In [ ]:
# ========== 把整本 Hamlet 追加进同一条 user content ==========
# 长静态文本是提示缓存实验的关键：前缀在多次请求间保持一致

question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet

In [ ]:
# ========== 带全文上下文再问一次（首次可能写入缓存） ==========

response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 观察 cached_tokens 与费用 ==========

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

In [ ]:
# ========== 立刻再请求一次：更容易看到缓存命中 ==========

response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 第二次长请求的用量 / 缓存 / 费用 ==========

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

## OpenAI 的提示缓存

文档：https://platform.openai.com/docs/guides/prompt-caching

> 缓存命中依赖提示的**精确前缀匹配**。想吃到缓存红利：把说明、示例等**静态内容放前面**，把用户可变信息放后面。图像与工具定义在请求间也必须一致。

缓存命中的输入通常便宜约 **4 倍**。  
定价：https://openai.com/api/pricing/


## Anthropic 的提示缓存

文档：https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching

- 你需要**显式标记**要缓存的内容  
- 「启动」缓存可能多付约 25%  
- 之后复用缓存输入可显著降价（文档称可到约 10 倍量级）

定价：https://www.anthropic.com/pricing#api


## Gemini 的提示缓存

Gemini 支持「隐式」与「显式」两种提示缓存。  
文档：https://ai.google.dev/gemini-api/docs/caching?lang=python


## 有趣实验：聊天机器人之间的对抗对话

你已经熟悉把提示组织成 `messages` 列表，例如：

```
[
    {"role": "system", "content": "这里是系统消息"},
    {"role": "user", "content": "此处是用户提示"}
]
```

同一结构也能承载更长历史：

```
[
    {"role": "system", "content": "这里是系统消息"},
    {"role": "user", "content": "第一轮用户"},
    {"role": "assistant", "content": "助手回复"},
    {"role": "user", "content": "新的用户提示"}
]
```

下面用两份历史列表，让两个模型互相把对方上轮输出当成 user 输入。


In [ ]:
# ========== 双模型对话：人设、模型名、初始台词 ==========
# system prompt 与模型 id 保留英文/原样；用较便宜型号控成本

# 让我们在 GPT-4.1-mini 和 Claude-haiku-4.5 之间进行对话
# 我们使用廉价版本的模型，因此成本将是最低的

gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5"

gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

claude_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

In [ ]:
# ========== call_gpt：把双方历史映成 GPT 视角的 messages ==========

def call_gpt():
    # 先放 GPT 的 system
    messages = [{"role": "system", "content": gpt_system}]
    # 对 GPT 来说：自己说过的是 assistant，Claude 说过的是 user
    for gpt, claude in zip(gpt_messages, claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
    # 请求下一句 GPT 回复
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content


In [ ]:
# 试跑一轮：看 GPT 在当前历史下会说什么
call_gpt()


In [ ]:
# ========== call_claude：映射历史 + 追加 GPT 最新一句为 user ==========

def call_claude():
    messages = [{"role": "system", "content": claude_system}]
    # 对 Claude：GPT 的话是 user，自己的话是 assistant
    for gpt, claude_message in zip(gpt_messages, claude_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": claude_message})
    # 再加上 GPT 最新一句，请 Claude 回应
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = anthropic.chat.completions.create(model=claude_model, messages=messages)
    return response.choices[0].message.content


In [ ]:
# 试跑 Claude 一轮
call_claude()


In [ ]:
# 再叫一次 GPT（历史若未 append，仍基于初始列表）
call_gpt()


In [ ]:
# ========== 多轮循环：交替调用并展示 Markdown ==========

# 重置开场白，避免沿用上格试跑残留
gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

# 先展示双方第一句
display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Claude:\n{claude_messages[0]}\n"))

# 进行 5 个来回：GPT 说完立刻 append，Claude 才能看到最新句
for i in range(5):
    # 请 GPT 基于当前双方历史生成下一句
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    # 写入 GPT 侧历史
    gpt_messages.append(gpt_next)
    
    # 请 Claude 回应 GPT 最新一句
    claude_next = call_claude()
    display(Markdown(f"### Claude:\n{claude_next}\n"))
    # 写入 Claude 侧历史
    claude_messages.append(claude_next)


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">继续之前</h2>
            <span style="color:#900;">
                请先真正弄懂上面的对话如何工作，尤其是 <code>messages</code> 列表如何被填充。
                需要时可加 <code>print</code>。然后试着改 system prompt 换个性——比如一个悲观、一个乐观。
            </span>
        </td>
    </tr>
</table>


# 更高级练习：三方对话

试着做成**三方**对话（例如再拉上 Gemini）。社区贡献目录里已有学生实现可参考。

较稳妥的提示结构：每次只给 **1 个 system + 1 个 user**，并在 user 里贴上**迄今完整对话记录**：

```python
system_prompt = """
你是 Alex，一个非常好争论的聊天机器人……
你正在与 Blake 和 Charlie 交谈。
"""

user_prompt = f"""
你是 Alex，正在与 Blake 和 Charlie 对话。
到目前为止的对话如下：
{conversation}
现在，作为 Alex，说出你接下来想说的话。
"""
```

先自己试，再看解决方案。用 OpenAI Python 客户端访问 Gemini（见上文兼容端点）通常最简单。

## 附加练习

也可以把其中一个角色换成 Ollama 本地开源模型。


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">业务相关性</h2>
            <span style="color:#181;">
                这种「消息列表」对话结构，正是构建对话式 AI 助手、并在多轮中保持上下文的核心。
                后续实验会用它做助手，你也可以迁移到自己的业务场景。
            </span>
        </td>
    </tr>
</table>


In [ ]:
# ========== 三方对话段：重新导入（可独立运行后半部分） ==========

# os：环境变量
import os
# requests：如需再探测 Ollama
import requests
# load_dotenv：加载 .env
from dotenv import load_dotenv
# OpenAI：指向本地 Ollama
from openai import OpenAI
# Markdown 展示三方发言
from IPython.display import Markdown, display


In [ ]:
# 加载 .env（本段主要用本地 Ollama，仍保持与课程一致的初始化）
load_dotenv(override=True)


In [ ]:
# Ollama 的 OpenAI 兼容基址（URL 勿改）
BASE_URL = "http://localhost:11434/v1"


In [ ]:
# ========== 共享对话笔录：三方都能在 user prompt 里看到 ==========

conversation = "Llama: Aha, there are two of you..."


In [ ]:
# ========== Llama 角色：客户端 + system/user 模板 ==========

# 三个角色都打到同一本地 Ollama OpenAI 兼容端点
llama = OpenAI(base_url=BASE_URL, api_key = "ollama")
# system：人设（保留英文，影响行为）
llama_system_prompt = """
You are Llama, a wife who is very angry when confront a cheating partner; 
you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are trapped in a three-way conversation with GPT (your cheating partner) and Qwen (the 'Third Party').

"""

# user：提醒模型身份，并注入迄今 conversation（f-string 会捕获当前变量）
llama_user_prompt =f"""
    The conversation so far is {conversation}. 
    Now with this, respond with what you would like to say next, as Llama. Don't try to act as other twos.
"""


In [ ]:
# ========== GPT 角色：不忠丈夫人设 ==========

# 同样指向本地 Ollama
gpt = OpenAI(base_url=BASE_URL, api_key = "ollama")
# system prompt 保留英文
gpt_system_prompt = """
You are GPT, the unfaithful husband in a high-stakes confrontation. 
You are caught between your spouse (Llama) and your side-partner (Qwen).
Try explain the situation in the meaningful way
"""

# user：要求降级冲突、保持对话格式、限制篇幅
gpt_user_prompt = f"""
    The conversation so far is: {conversation}.
    Now with this, respond with what you would like to say next, as GPT. 
    Keep the format of the conversation and try to de-escalate Llama's anger with annoying logic (not more than 500 words).
"""


In [ ]:
# ========== Qwen 角色：third party 人设 ==========

qwen = OpenAI(base_url=BASE_URL, api_key = "ollama")

# system：第三方角色（英文 prompt 勿改）
qwen_system_prompt = """
You are Qwen, the sophisticated the side chick in this confrontation. 
You are in a three-way fight with GPT (your partner) and Llama (the spouse).
You should challenge the wife constantly.

"""

# user：要求对 Llama 保持高傲语气，并附上对话笔录
qwen_user_prompt = f"""
    Now with this, respond with what you would like to say next, as Qwen. 
    Keep the format of the conversation and be as condescending to Llama as possible.
    The conversation so far is: {conversation}.
"""



In [ ]:
# ========== 三个本地模型名（需事先 ollama pull） ==========

llama_model = "llama3.2"
gpt_model = "gpt-oss:20b"
qwen_model = "qwen2.5-coder:14b"


In [ ]:
# ========== call_llama：system + user（user 里再拼 conversation） ==========

def call_llama():
    messages = [
        {"role": "system", "content": llama_system_prompt},
        # 原实现把 conversation 再拼一次；保留不改
        {"role": "user", "content": llama_user_prompt + conversation}
    ]
    # 调试：看实际发给模型的 messages
    print(messages)
    response = llama.chat.completions.create(model=llama_model, messages=messages)
    return response.choices[0].message.content


In [ ]:
# ========== call_gpt：同样的单轮 system+user 结构 ==========

def call_gpt():
    # 每次只发 1 system + 1 user（user 内含完整笔录）
    messages = [
        {"role": "system", "content": gpt_system_prompt},
        # 原实现再拼接 conversation；保留不改
        {"role": "user", "content": gpt_user_prompt + conversation}
    ]
    # 用 gpt_model（本地）生成下一句
    response = gpt.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content


In [ ]:
# ========== call_qwen：第三位发言者 ==========

def call_qwen():
    messages = [
        {"role": "system", "content": qwen_system_prompt},
        {"role": "user", "content": qwen_user_prompt + conversation}
    ]
    # 调用本地 qwen 模型
    response = qwen.chat.completions.create(model=qwen_model, messages=messages)
    return response.choices[0].message.content


In [ ]:
# 查看当前共享对话笔录
print(conversation)


In [ ]:
# 单独试 Llama 一句
call_llama()


In [ ]:
# 单独试 GPT 一句
call_gpt()


In [ ]:
# ========== 三方轮转 5 轮：发言后追加进 conversation ==========

for i in range(5):
    # Llama 先发言并 Markdown 展示
    llama_next = call_llama()
    display(Markdown(f"### Llama:\n{llama_next}\n"))
    # 追加进共享笔录，供后续角色看见
    conversation += llama_next + "\n"

    
    # GPT 接话
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    conversation += gpt_next + "\n"

    # Qwen 收尾本轮
    qwen_next = call_qwen()
    display(Markdown(f"### Qwen:\n{qwen_next}\n"))
    conversation += qwen_next + "\n"
